# CJM trên Kaggle: huấn luyện và đánh giá ReqGNN

**Trước khi chạy (thanh bên phải của notebook):**
1. *Settings → Accelerator*: **GPU T4** (tăng tốc embedding BGE-M3; phần huấn luyện ReqGNN chạy CPU).
2. *Settings → Internet*: **On** (clone code, tải BGE-M3 và dữ liệu ngoài). Cần xác minh số điện thoại cho tài khoản Kaggle.
3. *Add-ons → Secrets* (tùy chọn):
   - `GITHUB_TOKEN`: chỉ cần nếu repo GitHub ở chế độ private (token chỉ quyền đọc).
   - `LLM_API_KEY`: khóa Gemini. **Không cần** cho các bộ dữ liệu trong notebook này: `req_cache.json` đã chứa kết quả trích xuất của bộ chính, resume-job-description-fit (1119 CV, 351 JD) và vacancy (30 CV, 5 JD). Chỉ cần khi bạn thêm văn bản mới.

Code phải được **push lên GitHub** (nhánh `BRANCH` bên dưới) trước khi chạy, kể cả `models/req_model.pt`.

In [ ]:
import os, subprocess
REPO = "https://github.com/hoangphungvan/doantotnghiep.git"
BRANCH = "new-update-version"          # đổi thành nhánh bạn đã push
WORK = "/kaggle/working/cjm"           # /kaggle/input chỉ đọc, nên code chạy trong /kaggle/working

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    def secret(k):
        try:
            return _sec.get_secret(k)
        except Exception:
            return None
except ImportError:                     # chạy ngoài Kaggle
    secret = lambda k: os.environ.get(k)

tok = secret("GITHUB_TOKEN")
url = REPO.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORK], check=True)
    subprocess.run(["git", "-C", WORK, "remote", "set-url", "origin", REPO], check=True)  # không để token lại trong .git/config
os.chdir(WORK)
key = secret("LLM_API_KEY")
if key:
    os.environ["LLM_API_KEY"] = key
print("cwd:", os.getcwd(), "| Gemini key:", "có" if key else "không")

Cài thư viện còn thiếu. Kaggle đã có sẵn torch, sentence-transformers, pandas, scikit-learn. Không cần `torch-scatter` và `torch-sparse`, vì PyG dùng `scatter` thuần PyTorch.

In [ ]:
!pip install -q torch_geometric python-docx python-dotenv PyMuPDF openai
import torch, torch_geometric
print(torch.__version__, torch_geometric.__version__, "| GPU:", torch.cuda.is_available())

## Dữ liệu ngoài

Code tìm dữ liệu ngoài ở `data/raw/cnamuangtoun/` (resume-job-description-fit) và `data/raw/vacancy-resume-matching-dataset-main/vacancy-resume-matching-dataset-main/`.
Cell dưới tải hai bộ này từ Hugging Face và GitHub. Nếu bạn đã upload chúng thành Kaggle Dataset, hãy đặt `KAGGLE_INPUT` thành đường dẫn tương ứng trong `/kaggle/input/...` để dùng bản đó thay vì tải.

In [ ]:
import os, subprocess
from huggingface_hub import snapshot_download

CNA = "data/raw/cnamuangtoun"
VK = "data/raw/vacancy-resume-matching-dataset-main/vacancy-resume-matching-dataset-main"
KAGGLE_INPUT = {CNA: None, VK: None}   # vd {CNA: "/kaggle/input/resume-job-description-fit", ...}

os.makedirs(os.path.dirname(VK), exist_ok=True)
for dst, src in KAGGLE_INPUT.items():
    if src and not os.path.exists(dst):
        os.symlink(src, dst)
if not os.path.exists(CNA):
    snapshot_download("cnamuangtoun/resume-job-description-fit", repo_type="dataset", local_dir=CNA)
if not os.path.exists(VK):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/NataliaVanetik/vacancy-resume-matching-dataset.git", VK], check=True)
print(os.listdir(CNA), os.listdir(VK)[:5])

Kiểm tra nhanh: nhãn của bộ dữ liệu chính và phần diễn giải.

In [ ]:
!python -X utf8 tests/test_dataset_labels.py
!python -X utf8 tests/test_explain.py

## 1. Mô hình triển khai

Lệnh `final` chạy 5 fold theo JD (đo NDCG@10 và mức 0–3), fit ngưỡng, rồi huấn luyện lại trên toàn bộ dữ liệu và lưu `models/req_model.pt`.
Lần đầu sẽ tải BGE-M3 (khoảng 2,2 GB) và mã hóa mọi mục yêu cầu, cache vào `data/cache/req_emb_*.npz`.

In [ ]:
!python -X utf8 src/modeling/train_req.py final

## 2. Đánh giá trên bộ dữ liệu chính (5 seed): chia theo JD và loại trừ cả hai chiều

In [ ]:
CONFIGS = "main_pairbce,pair_skip_mlp_specfeat"
!python -X utf8 src/modeling/train_req.py jd --seeds 5 --configs {CONFIGS}
!python -X utf8 src/modeling/train_req.py double --seeds 5 --configs {CONFIGS}

## 3. Kiểm định ngoài

Kết quả trích xuất của cả hai bộ đã có sẵn trong `req_cache.json`, nên cell đầu chỉ cần khi bạn thay dữ liệu (cần `LLM_API_KEY`, không có khóa thì cell tự bỏ qua). Lần đầu sẽ mã hóa thêm vài nghìn mục bằng BGE-M3, chạy nhanh trên GPU.

In [ ]:
import os
if os.environ.get("LLM_API_KEY"):
    !python -X utf8 src/data_collection/extract_external.py
else:
    print("Không có LLM_API_KEY: chỉ dùng văn bản đã có trong cache")

In [ ]:
!python -X utf8 src/modeling/train_ext.py --protocol jd --seeds 5 --configs {CONFIGS}
!python -X utf8 src/modeling/train_vk.py --seeds 5 --configs {CONFIGS}

## 4. Gom kết quả để tải về (tab *Output* của notebook)

In [ ]:
import shutil, glob
os.makedirs("/kaggle/working/outputs", exist_ok=True)
for f in ["models/req_model.pt", *glob.glob("report/results/*.csv")]:
    shutil.copy(f, "/kaggle/working/outputs/")
shutil.make_archive("/kaggle/working/cjm_outputs", "zip", "/kaggle/working/outputs")
print(sorted(os.listdir("/kaggle/working/outputs")))